<a href="https://colab.research.google.com/github/chouihyunjun/-B-3-/blob/%EB%B0%95%EC%83%81%EC%95%84/%EC%82%AC%EC%9A%A9%EC%9E%90%EA%B0%90%EC%A0%95%EC%9D%B8%EC%8B%9D%EC%9D%8C%EC%95%85%EC%B6%94%EC%B2%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import InferenceClient

# 1. 토큰 로드
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    print("❌ 토큰 로드 실패: 코랩 Secrets를 확인하세요.")

# 2. 모델 설정
MODEL = "Qwen/Qwen2.5-7B-Instruct"
client = InferenceClient(token=HF_TOKEN)

# 💾 [핵심] 우리가 직접 준비한 진짜 유튜브 플레이리스트 주소록 (딕셔너리 데이터)
# 팀원들과 함께 좋은 링크를 수집해서 여기에 채워 넣으시면 됩니다!
MUSIC_DATABASE = {
    "기쁨": [
        {"title": "틀자마자 저절로 리듬타게 되는 뉴욕 플리🗽기분 확 살아나는 팝송 모음🎶 신나는 노동요🕺🏻", "url": "https://youtu.be/EIBTbf_AMx4?si=qGi5c1kH5_ZzktQ3"},
        {"title": "🚘 운전할 때 잠이 확 깨는 드라이브 팝송│💥 심장이 요동치는 플레이리스트🌿 기분 좋게 시작하는 하루", "url": "https://youtu.be/46iib26GrMM?si=Vb_bgdzx0N8seQAq"}
    ],
    "슬픔": [
        {"title": "비 오는 날 듣는 먹먹한 발라드 모음", "url": "https://www.youtube.com/watch?v=실제_유튜브_아이디3"},
        {"title": "위로가 필요할 때 잔잔하게 듣는 음악", "url": "https://www.youtube.com/watch?v=실제_유튜브_아이디4"}
    ],
    "분노": [
        {"title": "화날 때 다 부수는 락/메탈 플레이리스트", "url": "https://youtu.be/m9f5XgDLlWM?si=WyIvAU7zX24nnX9D"}
    ],
    "스트레스": [
        {"title": "스트레스 날려버릴 신나는 힙합/EDM", "url": "https://www.youtube.com/watch?v=실제_유튜브_아이디6"},
        {"title": "지친 퇴근길, 마음을 편하게 해주는 재즈", "url": "https://www.youtube.com/watch?v=실제_유튜브_아이디7"}
    ],
    "잔잔함": [
        {"title": "새벽 감성, 조용한 카페 가요 에센셜", "url": "https://www.youtube.com/watch?v=실제_유튜브_아이디8"},
        {"title": "공부하거나 일할 때 듣는 Lo-fi 비트 모음", "url": "https://www.youtube.com/watch?v=실제_유튜브_아이디9"}
    ]
}

# 3. AI 감정 분석 함수 (오직 감정 '단어' 하나만 쏙 뽑아내도록 지시)
def analyze_emotion(user_input: str):
    messages = [
        {
            "role": "system",
            "content": (
                "당신은 사용자의 문장을 분석하는 감정 분류기입니다.\n"
                "사용자의 입력 문장을 보고 오직 [기쁨, 슬픔, 분노, 스트레스, 잔잔함] 중 하나의 단어로만 답변하세요.\n"
                "수식어나 다른 설명은 절대 하지 말고 딱 한 단어만 출력하세요. 예: 기쁨"
            )
        },
        {
            "role": "user",
            "content": f"이 문장의 감정을 한 단어로 분류해줘: '{user_input}'"
        }
    ]

    response = client.chat_completion(messages=messages, model=MODEL, max_tokens=10)
    # AI 답변에서 공백을 제거하고 깨끗한 단어만 가져옴
    emotion = response.choices[0].message.content.strip()
    return emotion

# ──────────────────────────────────────────────────────────
# 4. 실행 루프
# ──────────────────────────────────────────────────────────
print("\n🎵 진짜 유튜브 링크를 매칭해주는 스마트 무드플레이리스트 시작!")
print("👉 종료하고 싶다면 '/quit'을 입력하세요.")

while True:
    유저_입력 = input("\n✍️ 오늘 당신의 기분은 어떤가요? > ").strip()

    if 유저_입력 == "/quit":
        print("\n👋 프로그램을 종료합니다.")
        break
    if not 유저_입력:
        continue

    print("🤖 AI가 감정을 분석하는 중입니다...")

    # AI가 '기쁨', '슬픔' 같은 단어 하나를 알아냅니다.
    감정결과 = analyze_emotion(유저_입력)
    print(f"🔍 AI 분석 결과 오늘 당신의 감정 상태는: [{감정결과}] 입니다.")

    print("\n🎵 [당신을 위한 실제 유튜브 추천 리스트] ──────────────────")
    # 우리가 만든 주소록(DATABASE)에서 해당 감정이 있는지 확인하고 매칭
    if 감정결과 in MUSIC_DATABASE:
        추천곡들 = MUSIC_DATABASE[감정결과]
        for 곡 in 추천곡들:
            print(f"▶️ 제목: {곡['title']}")
            print(f"🔗 링크: {곡['url']}\n")
    else:
        # AI가 이상한 단어를 뱉었을 때를 대비한 안전장치
        print("💡 감정에 맞는 플레이리스트를 찾지 못했습니다. 대신 검색 링크를 드릴게요!")
        print(f"🔗 링크: https://www.youtube.com/results?search_query={감정결과}+플레이리스트")
    print("───────────────────────────────────────────")


🎵 진짜 유튜브 링크를 매칭해주는 스마트 무드플레이리스트 시작!
👉 종료하고 싶다면 '/quit'을 입력하세요.

✍️ 오늘 당신의 기분은 어떤가요? > 기쁨
🤖 AI가 감정을 분석하는 중입니다...
🔍 AI 분석 결과 오늘 당신의 감정 상태는: [기쁨] 입니다.

🎵 [당신을 위한 실제 유튜브 추천 리스트] ──────────────────
▶️ 제목: 에너지 뿜뿜! 내적 댄스 유발 아이돌 탑백
🔗 링크: https://youtu.be/EIBTbf_AMx4?si=qGi5c1kH5_ZzktQ3

▶️ 제목: 드라이브할 때 듣는 청량한 팝송 플레이리스트
🔗 링크: https://www.youtube.com/watch?v=실제_유튜브_아이디2

───────────────────────────────────────────

✍️ 오늘 당신의 기분은 어떤가요? > 기쁨
🤖 AI가 감정을 분석하는 중입니다...
🔍 AI 분석 결과 오늘 당신의 감정 상태는: [기쁨] 입니다.

🎵 [당신을 위한 실제 유튜브 추천 리스트] ──────────────────
▶️ 제목: 에너지 뿜뿜! 내적 댄스 유발 아이돌 탑백
🔗 링크: https://youtu.be/EIBTbf_AMx4?si=qGi5c1kH5_ZzktQ3

▶️ 제목: 드라이브할 때 듣는 청량한 팝송 플레이리스트
🔗 링크: https://www.youtube.com/watch?v=실제_유튜브_아이디2

───────────────────────────────────────────

✍️ 오늘 당신의 기분은 어떤가요? > 기쁘다
🤖 AI가 감정을 분석하는 중입니다...
🔍 AI 분석 결과 오늘 당신의 감정 상태는: [기쁨] 입니다.

🎵 [당신을 위한 실제 유튜브 추천 리스트] ──────────────────
▶️ 제목: 에너지 뿜뿜! 내적 댄스 유발 아이돌 탑백
🔗 링크: https://youtu.be/EIBTbf_AMx4?si=qGi5c1kH5_ZzktQ3

▶️ 제목: 드라이브할 때 듣는 청량한 팝송 플레이리스